<a href="https://colab.research.google.com/github/voronv27/projects-in-ai-and-ml/blob/main/HW5/Homework5_Task3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Task 3

##Part 1:

We will implement the scaled dot-product attention discussed in class:

In [336]:
import numpy as np

def softmax(x):
  x = x - np.max(x, axis=-1, keepdims=True)
  return np.exp(x) / np.sum(np.exp(x), axis=-1, keepdims=True)

def scaled_dot_product_attention(Q, K, V):
  # following the steps from the lecture
  # 1) QK^T
  val = np.matmul(Q, K.T)

  # 2) scale (divide by sqrt(dk))
  dk = Q.shape[-1]
  val = val / np.sqrt(dk)

  # 3) softmax
  val = softmax(val)

  # 4) multiply by V
  val = np.matmul(val, V)

  return val

##Part 2:

Now we will integrate our `scaled_dot_product_attention` in the encoder architecture for an encoder-decoder seq2seq model. Unlike Bahdanau and Luong (discussed in class) which implement attention in the decoder, the homework document told us to implement it in the encoder, so that's what I did (hopefully that wasn't a misinterpretation on my part). Other than that, the dot-product style attention is mathematically similar to Luong attention since it's multiplicative.

(For full transparency, I am following this [tutorial](https://www.geeksforgeeks.org/machine-learning/seq2seq-model-in-machine-learning/) to build my encoder-decoder)

In [337]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [338]:
# making a pytorch version of the function from part 1,
# so that the gradients work in our seq2seq model
# I also added a mask for part 4, it's not used in this part
def scaled_dot_product_attention_torch(Q, K, V, mask=None):
  # following the steps from the lecture
  # 1) QK^T
  val = torch.matmul(Q, K.transpose(-2, -1))

  # 2) scale (divide by sqrt(dk))
  dk = Q.size(-1)
  val = val / torch.sqrt(torch.tensor(dk, dtype=torch.float32))

  # 3) softmax
  if mask is not None:
    val = val.masked_fill(mask == 0, -1e9)
  p_attn = F.softmax(val, dim=-1)

  # 4) multiply by V
  output = torch.matmul(p_attn, V)

  return output, p_attn

In [339]:
class Encoder(nn.Module):
  def __init__(self, input_dim, emb_dim, hidden_dim):
    super().__init__()
    self.embedding = nn.Embedding(input_dim, emb_dim)
    self.rnn = nn.GRU(emb_dim, hidden_dim)

  def forward(self, src):
    embedded = self.embedding(src)
    outputs, hidden = self.rnn(embedded)

    # add in our scaled dot-product attention func
    Q = hidden.permute(1, 0, 2)
    K = outputs.permute(1, 0, 2)
    V = K
    outputs, _ = scaled_dot_product_attention_torch(Q, K, V)
    outputs = outputs.permute(1, 0, 2)

    return outputs

In [340]:
class Decoder(nn.Module):
  def __init__(self, output_dim, emb_dim, hidden_dim):
    super().__init__()
    self.embedding = nn.Embedding(output_dim, emb_dim)
    self.rnn = nn.GRU(emb_dim, hidden_dim)
    self.fc = nn.Linear(hidden_dim, output_dim)

  def forward(self, input, hidden):
    input = input.unsqueeze(0)
    embedded = self.embedding(input)
    output, hidden = self.rnn(embedded, hidden)
    prediction = self.fc(output.squeeze(0))
    return prediction, hidden

In [341]:
class Seq2Seq(nn.Module):
  def __init__(self, encoder, decoder, device):
    super().__init__()
    self.encoder = encoder
    self.decoder = decoder
    self.device = device

  def forward(self, src, trg=None, teacher_forcing_ratio=0.5):
    batch_size = src.shape[1]
    trg_vocab_size = self.decoder.fc.out_features
    outputs = []

    hidden = self.encoder(src)

    input = torch.zeros(batch_size, dtype=torch.long).to(self.device)

    if trg is not None:
      max_len = trg.shape[0]
    else:
      max_len = 10

    for t in range(max_len):
      output, hidden = self.decoder(input, hidden)
      top1 = output.argmax(1)
      outputs.append(output.unsqueeze(0))

      if trg is not None and t < trg.shape[0] and torch.rand(1).item() < teacher_forcing_ratio:
        input = trg[t]
      else:
        input = top1

    outputs = torch.cat(outputs, dim=0)
    return outputs

##Part 3:

I chose to train my model on a subset of the Tatoeba dataset, downloaded from [here](https://www.manythings.org/anki/).


This data is licensed under CC-BY 2.0.  
For details, visit https://tatoeba.org/eng/terms_of_use

Attibution: www.manythings.org/anki and tatoeba.org

In [342]:
def load_data(path, max_samples=10000):
  pairs = []
  with open(path, encoding="utf-8") as f:
    for i, line in enumerate(f):
      if i >= max_samples:
        break
      src, trg = line.strip().split("\t")[:2]
      pairs.append((src.lower(), trg.lower()))
  return pairs

In [343]:
# 10k English to French data pairs
pairs = load_data("fra.txt")

In [344]:
from collections import Counter

def tokenize(sentence):
  return sentence.split()

def build_vocab(sentences, min_freq=2):
  counter = Counter()
  for s in sentences:
    counter.update(tokenize(s))

  vocab = {"<pad>":0, "<sos>":1, "<eos>":2, "<unk>":3}
  for word, freq in counter.items():
    if freq >= min_freq:
      vocab[word] = len(vocab)

  return vocab

def numericalize(sentence, vocab):
  tokens = ["<sos>"] + tokenize(sentence) + ["<eos>"]
  return [vocab.get(t, vocab["<unk>"]) for t in tokens]

In [345]:
from sklearn.model_selection import train_test_split

train_pairs, test_pairs = train_test_split(pairs, test_size=0.2, random_state=42)
src_train = [p[0] for p in train_pairs]
trg_train = [p[1] for p in train_pairs]
src_vocab = build_vocab(src_train)
trg_vocab = build_vocab(trg_train)

In [346]:
def prepare_data(pairs, src_vocab, trg_vocab):
  data = []
  for src, trg in pairs:
    src_ids = numericalize(src, src_vocab)
    trg_ids = numericalize(trg, trg_vocab)
    data.append((src_ids, trg_ids))
  return data

In [347]:
train_data = prepare_data(train_pairs, src_vocab, trg_vocab)
test_data = prepare_data(test_pairs, src_vocab, trg_vocab)

In [348]:
from torch.nn.utils.rnn import pad_sequence

def create_batches(data, batch_size, device):
  batches = []
  for i in range(0, len(data), batch_size):
    batch = data[i:i+batch_size]
    src_batch = [torch.tensor(pair[0], dtype=torch.long) for pair in batch]
    trg_batch = [torch.tensor(pair[1], dtype=torch.long) for pair in batch]

    src_batch = pad_sequence(src_batch, padding_value=0)  # seq_len x batch
    trg_batch = pad_sequence(trg_batch, padding_value=0)

    batches.append((src_batch.to(device), trg_batch.to(device)))

  return batches

In [349]:
import random
random.seed(42)

def train_epoch(model, data, optimizer, criterion, device):
  model.train()
  total_loss = 0

  data_copy = data.copy()
  random.shuffle(data_copy)
  batches = create_batches(data_copy, 64, device)

  for src, trg in batches:
    optimizer.zero_grad()

    output = model(src, trg)

    output = output[1:].view(-1, output.shape[-1])
    trg = trg[1:].view(-1)

    loss = criterion(output, trg)
    loss.backward()

    optimizer.step()
    total_loss += loss.item()

  return total_loss / len(data)

In [350]:
encoder = Encoder(len(src_vocab), 256, 512)
decoder = Decoder(len(trg_vocab), 256, 512)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = Seq2Seq(encoder, decoder, device=device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=0.0001)
model.to(device)

Seq2Seq(
  (encoder): Encoder(
    (embedding): Embedding(1587, 256)
    (rnn): GRU(256, 512)
  )
  (decoder): Decoder(
    (embedding): Embedding(2016, 256)
    (rnn): GRU(256, 512)
    (fc): Linear(in_features=512, out_features=2016, bias=True)
  )
)

In [351]:
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

def compute_bleu(model, data, trg_vocab, device):
  model.eval()
  scores = []

  inv_vocab = {v:k for k,v in trg_vocab.items()}
  smooth = SmoothingFunction().method1

  with torch.no_grad():
    for src_ids, trg_ids in data:
      src_ids = torch.tensor(src_ids, dtype=torch.long)
      src = src_ids.unsqueeze(1).to(device)

      pred = model(src)
      pred_tokens = [inv_vocab.get(int(t), "<unk>") for t in pred.squeeze().argmax(dim=-1)]

      ref_tokens = [inv_vocab.get(int(t), "<unk>") for t in trg_ids]

      score = sentence_bleu([ref_tokens], pred_tokens, smoothing_function=smooth)
      scores.append(score)

  return sum(scores) / len(scores)

In [352]:
epochs = 20
loss = 0
for i in range(epochs):
  loss = train_epoch(model, train_data, optimizer, nn.CrossEntropyLoss(), device)
  if (i % 5) == 0:
    print(f"Epoch {i}, train loss {loss:.4f}")
    bleu = compute_bleu(model, test_data, trg_vocab, device)
    print(f"Bleu score on test data: {bleu:.4f}")
    print()

print("Final Metrics:")
print(f"Final epoch {epochs}, train loss {loss:.4f}")
bleu = compute_bleu(model, test_data, trg_vocab, device)
print(f"Final Bleu score on test data: {bleu:.4f}")

Epoch 0, train loss 0.0419
Bleu score on test data: 0.0495

Epoch 5, train loss 0.0164
Bleu score on test data: 0.0800

Epoch 10, train loss 0.0104
Bleu score on test data: 0.0853

Epoch 15, train loss 0.0085
Bleu score on test data: 0.0903

Final Metrics:
Final epoch 20, train loss 0.0080
Final Bleu score on test data: 0.0965


The BLEU score for my model was 0.0965. Considering BLEU can go up to 1, this is pretty bad. Most likely, training on a small subset of data is what led to this result. We do see the model improve from its initial BLEU score, so we know that the training did help it become slightly better at translation from the untrained version.

##Part 4:

Finally, I will implement a simplified transformer model from scratch and apply it to the same translation task as in part 3.

(I referenced the original transformer in ["Attention is All You Need"](https://nlp.seas.harvard.edu/2018/04/03/attention.html) for this task)

###Model Code

In [402]:
import copy
def clones(module, N):
  "Produce N identical layers."
  return nn.ModuleList([copy.deepcopy(module) for _ in range(N)])

In [403]:
class LayerNorm(nn.Module):
  "Construct a layernorm module (See citation for details)."
  def __init__(self, features, eps=1e-6):
    super(LayerNorm, self).__init__()
    self.a_2 = nn.Parameter(torch.ones(features))
    self.b_2 = nn.Parameter(torch.zeros(features))
    self.eps = eps

  def forward(self, x):
    mean = x.mean(-1, keepdim=True)
    std = x.std(-1, keepdim=True)
    return self.a_2 * (x - mean) / (std + self.eps) + self.b_2

In [404]:
class Encoder(nn.Module):
  "Core encoder is a stack of N layers"
  def __init__(self, layer, N):
    super(Encoder, self).__init__()
    self.layers = clones(layer, N)
    self.norm = LayerNorm(layer.size)

  def forward(self, x, mask):
    "Pass the input (and mask) through each layer in turn."
    for layer in self.layers:
      x = layer(x, mask)
    return self.norm(x)

In [405]:
class SublayerConnection(nn.Module):
  """
  A residual connection followed by a layer norm.
  Note for code simplicity the norm is first as opposed to last.
  """
  def __init__(self, size, dropout):
    super(SublayerConnection, self).__init__()
    self.norm = LayerNorm(size)
    self.dropout = nn.Dropout(dropout)

  def forward(self, x, sublayer):
    "Apply residual connection to any sublayer with the same size."
    return x + self.dropout(sublayer(self.norm(x)))

In [406]:
class EncoderLayer(nn.Module):
  "Encoder is made up of self-attn and feed forward (defined below)"
  def __init__(self, size, self_attn, feed_forward, dropout):
    super(EncoderLayer, self).__init__()
    self.self_attn = self_attn
    self.feed_forward = feed_forward
    self.sublayer = clones(SublayerConnection(size, dropout), 2)
    self.size = size

  def forward(self, x, mask):
    x = self.sublayer[0](x, lambda x: self.self_attn(x, x, x, mask))
    return self.sublayer[1](x, self.feed_forward)

In [407]:
class Decoder(nn.Module):
  "Generic N layer decoder with masking."
  def __init__(self, layer, N):
    super(Decoder, self).__init__()
    self.layers = clones(layer, N)
    self.norm = LayerNorm(layer.size)

  def forward(self, x, memory, src_mask, tgt_mask):
    for layer in self.layers:
      x = layer(x, memory, src_mask, tgt_mask)
    return self.norm(x)

In [408]:
class DecoderLayer(nn.Module):
  "Decoder is made of self-attn, src-attn, and feed forward (defined below)"
  def __init__(self, size, self_attn, src_attn, feed_forward, dropout):
    super(DecoderLayer, self).__init__()
    self.size = size
    self.self_attn = self_attn
    self.src_attn = src_attn
    self.feed_forward = feed_forward
    self.sublayer = clones(SublayerConnection(size, dropout), 3)

  def forward(self, x, memory, src_mask, tgt_mask):
    m = memory
    x = self.sublayer[0](x, lambda x: self.self_attn(x, x, x, tgt_mask))
    x = self.sublayer[1](x, lambda x: self.src_attn(x, m, m, src_mask))
    return self.sublayer[2](x, self.feed_forward)

In [409]:
class EncoderDecoder(nn.Module):
  """
  A standard Encoder-Decoder architecture. Base for this and many
  other models.
  """
  def __init__(self, encoder, decoder, src_embed, tgt_embed, generator):
    super(EncoderDecoder, self).__init__()
    self.encoder = encoder
    self.decoder = decoder
    self.src_embed = src_embed
    self.tgt_embed = tgt_embed
    self.generator = generator

  def forward(self, src, tgt, src_mask, tgt_mask):
    "Take in and process masked src and target sequences."
    return self.decode(self.encode(src, src_mask), src_mask,
                       tgt, tgt_mask)

  def encode(self, src, src_mask):
    return self.encoder(self.src_embed(src), src_mask)

  def decode(self, memory, src_mask, tgt, tgt_mask):
    return self.decoder(self.tgt_embed(tgt), memory, src_mask, tgt_mask)

In [410]:
class Generator(nn.Module):
  def __init__(self, d_model, vocab):
    super(Generator, self).__init__()
    self.proj = nn.Linear(d_model, vocab)

  def forward(self, x):
    return F.log_softmax(self.proj(x), dim=-1)

In [411]:
def subsequent_mask(size):
  "Mask out subsequent positions."
  attn_shape = (1, size, size)
  subsequent_mask = np.triu(np.ones(attn_shape), k=1).astype('uint8')
  return torch.from_numpy(subsequent_mask) == 0

In [412]:
def attention(Q, K, V, mask=None, dropout=None):
  # uses our dot product attention function from before
  # but adds in mask and dropout
  output, p_attn = scaled_dot_product_attention_torch(Q, K, V, mask)
  if dropout is not None:
    output = dropout(output)
  return output, p_attn

In [413]:
class MultiHeadedAttention(nn.Module):
  def __init__(self, h, d_model, dropout=0.1):
    "Take in model size and number of heads."
    super(MultiHeadedAttention, self).__init__()
    assert d_model % h == 0
    # We assume d_v always equals d_k
    self.d_k = d_model // h
    self.h = h
    self.linears = clones(nn.Linear(d_model, d_model), 4)
    self.attn = None
    self.dropout = nn.Dropout(p=dropout)

  def forward(self, query, key, value, mask=None):
    "Implements Figure 2"
    if mask is not None:
      # Same mask applied to all h heads.
      mask = mask.unsqueeze(1)
    nbatches = query.size(0)

    # 1) Do all the linear projections in batch from d_model => h x d_k
    query, key, value = \
      [l(x).view(nbatches, -1, self.h, self.d_k).transpose(1, 2)
       for l, x in zip(self.linears, (query, key, value))]

    # 2) Apply attention on all the projected vectors in batch.
    x, self.attn = attention(query, key, value, mask=mask,
                             dropout=self.dropout)

    # 3) "Concat" using a view and apply a final linear.
    x = x.transpose(1, 2).contiguous() \
        .view(nbatches, -1, self.h * self.d_k)
    return self.linears[-1](x)

In [414]:
class PositionwiseFeedForward(nn.Module):
  "Implements FFN equation."
  def __init__(self, d_model, d_ff, dropout=0.1):
    super(PositionwiseFeedForward, self).__init__()
    self.w_1 = nn.Linear(d_model, d_ff)
    self.w_2 = nn.Linear(d_ff, d_model)
    self.dropout = nn.Dropout(dropout)

  def forward(self, x):
    return self.w_2(self.dropout(F.relu(self.w_1(x))))

In [415]:
class Embeddings(nn.Module):
  def __init__(self, d_model, vocab):
    super(Embeddings, self).__init__()
    self.lut = nn.Embedding(vocab, d_model)
    self.d_model = d_model

  def forward(self, x):
    return self.lut(x) * np.sqrt(self.d_model)

In [416]:
from torch.autograd import Variable
class PositionalEncoding(nn.Module):
  "Implement the PE function."
  def __init__(self, d_model, dropout, max_len=5000):
    super(PositionalEncoding, self).__init__()
    self.dropout = nn.Dropout(p=dropout)

    # Compute the positional encodings once in log space.
    pe = torch.zeros(max_len, d_model)
    position = torch.arange(0, max_len).unsqueeze(1)
    div_term = torch.exp(torch.arange(0, d_model, 2) *
                             -(np.log(10000.0) / d_model))
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    pe = pe.unsqueeze(0)
    self.register_buffer('pe', pe)

  def forward(self, x):
    x = x + Variable(self.pe[:, :x.size(1)],
                     requires_grad=False)
    return self.dropout(x)

In [417]:
def make_model(src_vocab, tgt_vocab, N=2,
               d_model=64, d_ff=128, h=2, dropout=0.1):
  "Helper: Construct a model from hyperparameters."
  c = copy.deepcopy
  attn = MultiHeadedAttention(h, d_model)
  ff = PositionwiseFeedForward(d_model, d_ff, dropout)
  position = PositionalEncoding(d_model, dropout)
  model = EncoderDecoder(
    Encoder(EncoderLayer(d_model, c(attn), c(ff), dropout), N),
    Decoder(DecoderLayer(d_model, c(attn), c(attn),
                         c(ff), dropout), N),
    nn.Sequential(Embeddings(d_model, src_vocab), c(position)),
    nn.Sequential(Embeddings(d_model, tgt_vocab), c(position)),
    Generator(d_model, tgt_vocab))

  # This was important from their code.
  # Initialize parameters with Glorot / fan_avg.
  for p in model.parameters():
    if p.dim() > 1:
      nn.init.xavier_uniform_(p)
  return model

###Training Code

We can reuse the tokenizer from part 3, as it is word-level (which meets the tokenization simplification criteria of this task).

As such, our dataset has already been processed. We just need to write the code to train this model.

In [418]:
class Batch:
  "Object for holding a batch of data with mask during training."
  def __init__(self, src, trg=None, pad=0):
    self.src = src
    self.src_mask = (src != pad).unsqueeze(-2)
    if trg is not None:
      self.trg = trg[:, :-1]
      self.trg_y = trg[:, 1:]
      self.trg_mask = \
        self.make_std_mask(self.trg, pad)
      self.ntokens = (self.trg_y != pad).data.sum()

  @staticmethod
  def make_std_mask(tgt, pad):
    "Create a mask to hide padding and future words."
    tgt_mask = (tgt != pad).unsqueeze(-2)
    tgt_mask = tgt_mask & Variable(
      subsequent_mask(tgt.size(-1)).type_as(tgt_mask.data))
    return tgt_mask

In [419]:
global max_src_in_batch, max_tgt_in_batch
def batch_size_fn(new, count, sofar):
  "Keep augmenting batch and calculate total number of tokens + padding."
  global max_src_in_batch, max_tgt_in_batch
  if count == 1:
    max_src_in_batch = 0
    max_tgt_in_batch = 0
  max_src_in_batch = max(max_src_in_batch,  len(new.src))
  max_tgt_in_batch = max(max_tgt_in_batch,  len(new.trg) + 2)
  src_elements = count * max_src_in_batch
  tgt_elements = count * max_tgt_in_batch
  return max(src_elements, tgt_elements)

In [420]:
def data_to_batch(data, batch_size, device):
  for i in range(0, len(data), batch_size):
    batch = data[i:i+batch_size]
    src_batch = [torch.tensor(x, dtype=torch.long) for x, y in batch]
    trg_batch = [torch.tensor(y, dtype=torch.long) for x, y in batch]

    src_batch = pad_sequence(src_batch, batch_first=True, padding_value=0).to(device)
    trg_batch = pad_sequence(trg_batch, batch_first=True, padding_value=0).to(device)

    yield Batch(src_batch, trg_batch)

In [421]:
import time

def run_epoch_transformer(data_iter, model, loss_compute):
  "Standard Training and Logging Function"
  start = time.time()
  total_tokens = 0
  total_loss = 0
  tokens = 0
  for i, batch in enumerate(data_iter):
    out = model.forward(batch.src, batch.trg,
                        batch.src_mask, batch.trg_mask)
    loss = loss_compute(out, batch.trg_y, batch.ntokens)
    total_loss += loss
    total_tokens += batch.ntokens
    tokens += batch.ntokens
    if i % 50 == 1:
      elapsed = time.time() - start
      print("Epoch Step: %d Loss: %f Tokens per Sec: %f" %
            (i, loss / batch.ntokens, tokens / elapsed))
      start = time.time()
      tokens = 0
  return total_loss / total_tokens

In [422]:
class SimpleLossCompute:
  "A simple loss compute and train function."
  def __init__(self, generator, criterion, opt=None):
    self.generator = generator
    self.criterion = criterion
    self.opt = opt

  def __call__(self, x, y, norm):
    x = self.generator(x)
    loss = self.criterion(x.contiguous().view(-1, x.size(-1)),
                          y.contiguous().view(-1)) / norm
    loss.backward()
    if self.opt is not None:
      self.opt.step()
      self.opt.optimizer.zero_grad()
    return loss.item() * norm

In [423]:
class NoamOpt:
  "Optim wrapper that implements rate."
  def __init__(self, model_size, factor, warmup, optimizer):
    self.optimizer = optimizer
    self._step = 0
    self.warmup = warmup
    self.factor = factor
    self.model_size = model_size
    self._rate = 0

  def step(self):
    "Update parameters and rate"
    self._step += 1
    rate = self.rate()
    for p in self.optimizer.param_groups:
      p['lr'] = rate
    self._rate = rate
    self.optimizer.step()

  def rate(self, step = None):
    "Implement `lrate` above"
    if step is None:
      step = self._step
    return self.factor * \
      (self.model_size ** (-0.5) *
      min(step ** (-0.5), step * self.warmup ** (-1.5)))

def get_std_opt(model):
  return NoamOpt(model.src_embed[0].d_model, 2, 4000,
                 torch.optim.Adam(model.parameters(), lr=0, betas=(0.9, 0.98), eps=1e-9))

In [424]:
class LabelSmoothing(nn.Module):
  "Implement label smoothing."
  def __init__(self, size, padding_idx, smoothing=0.0):
    super(LabelSmoothing, self).__init__()
    self.criterion = nn.KLDivLoss(reduction='sum')
    self.padding_idx = padding_idx
    self.confidence = 1.0 - smoothing
    self.smoothing = smoothing
    self.size = size
    self.true_dist = None

  def forward(self, x, target):
    assert x.size(1) == self.size
    true_dist = x.data.clone()
    true_dist.fill_(self.smoothing / (self.size - 2))
    true_dist.scatter_(1, target.data.unsqueeze(1), self.confidence)
    true_dist[:, self.padding_idx] = 0
    mask = torch.nonzero(target.data == self.padding_idx)
    if mask.dim() > 0:
      true_dist.index_fill_(0, mask.squeeze(), 0.0)
    self.true_dist = true_dist
    return self.criterion(x, Variable(true_dist, requires_grad=False))

In [426]:
model = make_model(len(src_vocab), len(trg_vocab))
model.to(device)
criterion = LabelSmoothing(size=len(trg_vocab), padding_idx=0, smoothing=0.0)
model_opt = NoamOpt(model.src_embed[0].d_model, 1, 400,
                    torch.optim.Adam(model.parameters(), lr=0, betas=(0.9, 0.98), eps=1e-9))

for epoch in range(10):
  train_iter = data_to_batch(train_data, 64, device)
  test_iter = data_to_batch(test_data, 64, device)

  model.train()
  run_epoch_transformer(train_iter, model,
                        SimpleLossCompute(model.generator, criterion, model_opt))
  model.eval()
  print(run_epoch_transformer(test_iter, model,
                              SimpleLossCompute(model.generator, criterion, None)))

Epoch Step: 1 Loss: 7.650344 Tokens per Sec: 9875.761719
Epoch Step: 51 Loss: 5.972209 Tokens per Sec: 10708.545898
Epoch Step: 101 Loss: 4.572917 Tokens per Sec: 10982.741211
Epoch Step: 1 Loss: 3.960365 Tokens per Sec: 12054.067383
tensor(3.8356, device='cuda:0')
Epoch Step: 1 Loss: 3.856628 Tokens per Sec: 10479.392578
Epoch Step: 51 Loss: 3.593283 Tokens per Sec: 10809.921875
Epoch Step: 101 Loss: 3.230530 Tokens per Sec: 10378.133789
Epoch Step: 1 Loss: 2.926494 Tokens per Sec: 11222.134766
tensor(2.7369, device='cuda:0')
Epoch Step: 1 Loss: 2.885505 Tokens per Sec: 10455.402344
Epoch Step: 51 Loss: 2.771891 Tokens per Sec: 10870.219727
Epoch Step: 101 Loss: 2.584885 Tokens per Sec: 10949.929688
Epoch Step: 1 Loss: 2.354893 Tokens per Sec: 12099.040039
tensor(2.2166, device='cuda:0')
Epoch Step: 1 Loss: 2.370585 Tokens per Sec: 10315.525391
Epoch Step: 51 Loss: 2.281659 Tokens per Sec: 10829.072266
Epoch Step: 101 Loss: 2.104633 Tokens per Sec: 10673.830078
Epoch Step: 1 Loss: 1.9

###Evaluation:

In [436]:
def get_sequence(seq, inv_vocab, eos_idx, pad_idx=0):
  tokens = []
  for t in seq:
    if t == eos_idx:
      break
    if t == pad_idx:
      continue
    tokens.append(inv_vocab.get(int(t), "<unk>"))
  return tokens

def compute_bleu_transformer(model, data, trg_vocab, device):
  model.eval()
  inv_vocab = {v:k for k,v in trg_vocab.items()}
  smooth = SmoothingFunction().method1
  scores = []

  with torch.no_grad():
    for batch in data:
      src = batch.src
      trg = batch.trg_y

      batch_size = src.size(0)
      sos_idx = trg_vocab["<sos>"]
      ys = torch.full((batch_size, 1), sos_idx, dtype=torch.long, device=device)

      max_len = trg.size(1)
      eos_idx = trg_vocab["<eos>"]

      for i in range(max_len):
        out = model(src, ys, batch.src_mask, Batch.make_std_mask(ys, pad=0))
        prob = model.generator(out[:, -1])
        _, next_word = torch.max(prob, dim=-1)
        next_word = next_word.unsqueeze(1)
        ys = torch.cat([ys, next_word], dim=-1)

        if (next_word == eos_idx).all():
          break

      for i in range(batch_size):
        pred_tokens = get_sequence(ys[i, 1:], inv_vocab, eos_idx)
        ref_tokens = get_sequence(trg[i], inv_vocab, eos_idx)
        score = sentence_bleu([ref_tokens], pred_tokens, smoothing_function=smooth)
        scores.append(score)
    return sum(scores) / len(scores)

In [437]:
model.eval()
print("Final Metrics:")
test_iter = data_to_batch(test_data, 64, device)
bleu = compute_bleu_transformer(model, test_iter, trg_vocab, device)
print(f"Final Bleu score on test data: {bleu:.4f}")

Final Metrics:
Final Bleu score on test data: 0.2471


The transformer performed significantly better than the Seq2Seq model in part 2/3, even when training on the exact same dataset. The improved performance is due to the differing architectures between Seq2Seq and Transformers.

Transformers use global attention instead of sequential memory. That is, rather than the seq2seq encoder compressing sentences into a single hidden state, the transformer uses self-attention over the entire sequence at every layer, so each token can directly attend to the other tokens. This gives the transformer a better contextual understanding and see the relationships between words regardless of the sequence length/distance between the words.

In addition, the transformer has multi-head attention. Unlike Seq2Seq, the transformer model had 2 attention heads. These two heads allow for multiple "perspectives" on the data (learn different things for things like positional relationships), so the model ends up with a better understanding.

My transformer also made use of residual connections and layer normalization, which stablized training and made sure gradients wouldn't vanish or explode.

It also should be noted that the transformer model I implemented (based off of the original transformer) had specific hyperparameters which I copied over, and those hyperparameters were most likely some of the "best" parameters for the model. This is as opposed to my Seq2Seq model, where I did some tuning, but didn't look as extensively to find the optimal parameters, putting the model at a disadvantage.

For training speed, I saw that the transformer was able to be trained faster. This makes sense, since they process the input sequence simulaneously which allows for parallelization, and I ran this on GPU.